# 3_0 — Полная оценка: CV + LOO, распределения и графики

**Один ноутбук вместо 3_1…3_8.** Всё считается на честной кросс-валидации по площадкам
(каждую площадку тестирует модель, которая её не видела) — а не на одном случайном сплите.

**Что внутри:** лидерборд с разбросом по фолдам · мультипанельные графики метрик · значимость
(bootstrap + Wilcoxon/Holm) · калибровка/покрытие · P³/Pareto (физ. допустимость) · разрез по
режиму/грунту · абляции · OOD (без префикса) · A/B flow-vs-gaussian · манифест воспроизводимости.
Все подписи графиков — на английском.

**Как запускать** (тумблеры в ячейке ниже):
- быстрый headline: `QUICK=False`, `N_REPEATS=1`, `NESTED=True`, `RUN_LOO=RUN_AB=False`;
- полный отчёт: всё `True`, `N_REPEATS=3` (дорого — часы/дни, лучше GPU).

Устройство выбирается само (CUDA → Apple Metal → CPU); форс — `LIQ_DEVICE=cpu`.
**Git-проверок нет** — прогон запускается всегда; preflight только печатает совет, если конфиг неполный.


In [1]:
import os, sys, json, time
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')): REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src')); os.chdir(REPO)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from liquefaction_ai.viz import register_theme, new_figure, save_figure, QUALITATIVE
register_theme()
print('REPO =', REPO)

REPO = /Users/nikita/Desktop/projects/liquefaction-ai


In [2]:
# ---- Run control ----
PUBLICATION_RUN = False   # False isolates all smoke outputs under results/smoke
QUICK          = False  # publication mode must never use demo epochs
N_SPLITS       = 5       # grouped-CV folds
N_REPEATS      = 3       # repeated CV (>=3): stability; primary = one full OOF pass
NESTED         = True    # fold-local hyperparameter selection (honest nested CV)
RUN_LOO        = True    # secondary: leave-one-site-out (expensive)
RUN_ABLATIONS  = True    # DPI-Flow component ablations across folds
RUN_OOD        = True    # OOD no-prefix stress on fold 0
RUN_AB         = True    # A/B conditional flow vs Gaussian posterior (pooled OOF; slow)
ABLATION_SEEDS = (42, 1042, 2042)
REF   = 'DPI-Flow'       # reference model for significance / forest plot
SAVE  = True
# Structured models are physically admissible (PVR=0) -> highlighted; flexible baselines -> muted.
PHYSICS = {'DPI-Flow', 'DPI-EVT', 'EVT-NeuralSSM'}
C_PHYS, C_BASE, C_REF = '#2f7ab5', '#8f6fae', '#e08214'

import torch
from liquefaction_ai import resolve_device, configure_performance, describe_device
device = configure_performance(resolve_device())
RUN_ROOT = os.path.join(REPO, 'results' if PUBLICATION_RUN else 'results/smoke')
TABLES = os.path.join(RUN_ROOT, 'analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB    = os.path.join(RUN_ROOT, 'tables');          os.makedirs(PUB, exist_ok=True)
FIGS   = os.path.join(RUN_ROOT, 'analysis_figs');   os.makedirs(FIGS, exist_ok=True)
SOURCE_GIT_CLEAN = False
if PUBLICATION_RUN:
    from liquefaction_ai.evaluation import publication_preflight
    publication_preflight(REPO, quick=QUICK, nested=NESTED, run_loo=RUN_LOO,
                          run_ablations=RUN_ABLATIONS, run_ood=RUN_OOD, run_ab=RUN_AB,
                          n_repeats=N_REPEATS, ablation_seeds=ABLATION_SEEDS, output_root=RUN_ROOT,
                          require_clean=False)
    SOURCE_GIT_CLEAN = True
from liquefaction_ai import load_population_artifact
pop, config = load_population_artifact('data/dataset')
meta = pop['meta']
_site = meta['site_id'] if 'site_id' in meta.columns else meta['object']
print(f"samples={len(meta)} | objects={meta['object'].nunique()} | sites={_site.nunique()} "
      f"| liq={int((meta['liq_label']>0.5).sum()) if 'liq_label' in meta.columns else '?'}")
print(f"mode={'PUBLICATION' if PUBLICATION_RUN else 'SMOKE'} device={device} QUICK={QUICK} "
      f"N_SPLITS={N_SPLITS} N_REPEATS={N_REPEATS} NESTED={NESTED} "
      f"mc_samples_eval={getattr(config,'mc_samples_eval',1)} outputs={RUN_ROOT}")

samples=812 | objects=14 | sites=13 | liq=490
mode=SMOKE device=mps QUICK=False N_SPLITS=5 N_REPEATS=3 NESTED=True mc_samples_eval=8 outputs=/Users/nikita/Desktop/projects/liquefaction-ai/results/smoke


## 1. Grouped repeated object-CV (primary estimand)

`evaluate_fold` trains all models on the fold train, calibrates σ on val and computes the FULL metric
set on the held-out test. `raw` = one row per (fold × model); `samples` = per-sample OOF predictions.

In [ ]:
from liquefaction_ai.evaluation.cross_validation import (build_folds, evaluate_fold, aggregate_cv,
                                                           aggregate_object_conformal, nliq_tail_tables)
folds = build_folds(meta, config, seed=42, loo=False, n_splits=N_SPLITS, n_repeats=N_REPEATS)
raw, samples = [], []
t0 = time.time()
for gid, fold in enumerate(folds):
    r, s = evaluate_fold(pop, config, fold, gid, device, quick=QUICK, nested=NESTED,
                         p3_reference=REF)
    raw.append(r); samples.append(s)
    print(f"rep{fold['repeat']} fold{fold['fold']} ({time.time()-t0:.0f}s): "
          + ' '.join(f"{m}={v:.3f}" for m, v in zip(r.model, r.Traj_RMSE_continuation.round(3))))
raw = pd.concat(raw, ignore_index=True); samples = pd.concat(samples, ignore_index=True)
raw.to_csv(os.path.join(TABLES, 'cv_grouped_raw.csv'), index=False)
samples.to_csv(os.path.join(TABLES, 'cv_grouped_samples.csv'), index=False)
print('done: raw', raw.shape, '| samples', samples.shape)

loo_raw = loo_samples = None
if RUN_LOO:
    loo_folds = build_folds(meta, config, seed=42, loo=True)
    _lr, _ls = [], []
    for gid, fold in enumerate(loo_folds):
        r, s = evaluate_fold(pop, config, fold, gid, device, quick=QUICK, nested=NESTED,
                             p3_reference=REF)
        _lr.append(r); _ls.append(s)
        print(f"LOO {gid + 1}/{len(loo_folds)} site={fold['test_object']} ({time.time()-t0:.0f}s)")
    loo_raw = pd.concat(_lr, ignore_index=True); loo_samples = pd.concat(_ls, ignore_index=True)
    loo_raw.to_csv(os.path.join(TABLES, 'cv_loo_raw.csv'), index=False)
    loo_samples.to_csv(os.path.join(TABLES, 'cv_loo_samples.csv'), index=False)
    print('LOO done:', loo_raw.shape, '| samples', loo_samples.shape)

In [ ]:
# Primary estimand = one full OOF pass (min repeat). Other repeats = stability only.
primary_raw = raw[raw['repeat'] == raw['repeat'].min()].copy()
summary = aggregate_cv(primary_raw); summary.to_csv(os.path.join(TABLES, 'cv_grouped_summary.csv'), index=False)
aggregate_cv(raw).to_csv(os.path.join(TABLES, 'cv_grouped_stability.csv'), index=False)
objconf = aggregate_object_conformal(samples, level=0.90)
objconf.to_csv(os.path.join(TABLES, 'cv_object_conformal.csv'), index=False)
tail_cases, tail_summary = nliq_tail_tables(samples, top_k_per_model=20, catastrophic_cycles=1000.0)
tail_cases.to_csv(os.path.join(PUB, 'nliq_tail_cases.csv'), index=False)
tail_summary.to_csv(os.path.join(TABLES, 'nliq_tail_summary.csv'), index=False)
display(tail_summary.round(3))

loo_summary = loo_objconf = None
if loo_raw is not None:
    loo_summary = aggregate_cv(loo_raw)
    loo_summary.to_csv(os.path.join(TABLES, 'cv_loo_summary.csv'), index=False)
    loo_objconf = aggregate_object_conformal(loo_samples, level=0.90)
    loo_objconf.to_csv(os.path.join(TABLES, 'cv_loo_object_conformal.csv'), index=False)

# Leaderboard: fold mean +/- SD (primary), sorted by post-prefix trajectory RMSE.
SHOW = ['Traj_RMSE_continuation', 'Traj_RMSE_worst', 'N_liq_logMAE', 'AUPRC', 'Brier',
        'ECE', 'Coverage_90', 'Physics_Violation_Rate', 'N_liq_Curve_Coherence_MAE',
        'Risk_Curve_Coherence_Rate']
g = primary_raw.groupby('model')
lb = pd.DataFrame({m: g[m].mean().round(4).astype(str) + ' ±' + g[m].std().round(3).astype(str)
                   for m in SHOW if m in primary_raw.columns})
lb = lb.reindex(g['Traj_RMSE_continuation'].mean().sort_values().index)
display(lb)

## 2. Metric figures (mean ± std across folds)

Point estimates on 13–14 sites are unstable, so we show the **full spread** across folds — a grouped
bar panel (mean ± fold std) and box plots, all on a single canvas each. Physics-admissible models
(PVR=0) are highlighted; flexible baselines are muted.

In [ ]:
def _bar_color(m):
    return C_REF if m == REF else (C_PHYS if m in PHYSICS else C_BASE)

# --- Multi-panel grouped bars: mean +/- fold std, one subplot per metric, on one canvas ---
panel = [('Traj_RMSE_continuation', True, 'Post-prefix trajectory RMSE'),
         ('Traj_RMSE_worst', True, 'Worst-state trajectory RMSE'),
         ('N_liq_logMAE', True, 'N_liq log-MAE (censored)'),
         ('AUPRC', False, 'Onset AUPRC'),
         ('Brier', True, 'Onset Brier'),
         ('ECE', True, 'Calibration error (ECE)'),
         ('Coverage_90', False, 'Coverage@90 (target 0.90)')]
panel = [p for p in panel if p[0] in raw.columns]
ncol = 4; nrow = int(np.ceil(len(panel) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 3.1 * nrow))
axes = np.atleast_1d(axes).ravel()
gm = raw.groupby('model')
for ax, (mtr, lower_better, title) in zip(axes, panel):
    mu = gm[mtr].mean(); sd = gm[mtr].std().fillna(0.0)
    order = mu.sort_values(ascending=lower_better).index.tolist()
    xs = np.arange(len(order))
    ax.bar(xs, mu[order].values, yerr=sd[order].values, capsize=3,
           color=[_bar_color(m) for m in order], edgecolor='white', linewidth=0.6, error_kw=dict(lw=1.3))
    if mtr == 'Coverage_90': ax.axhline(0.90, ls='--', color='#c46b6b', lw=1.0)
    ax.set_xticks(xs); ax.set_xticklabels(order, rotation=40, ha='right', fontsize=7)
    ax.set_title(title, fontsize=9.5); ax.set_ylabel(mtr, fontsize=8)
    ax.margins(x=0.02)
for ax in axes[len(panel):]: ax.set_visible(False)
fig.suptitle('Cross-validated metrics — mean ± fold std  (blue=physics, orange=reference, purple=baselines)',
             fontsize=12, y=1.005)
fig.tight_layout()
fig.savefig(os.path.join(FIGS, '3_0_metric_bars.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(FIGS, '3_0_metric_bars.pdf'), bbox_inches='tight')
display(fig)

In [ ]:
# --- Multi-panel box plots: full per-fold distribution, one canvas ---
box_metrics = [m for m, *_ in panel][:6]
fig, axes = plt.subplots(2, 3, figsize=(13.5, 6.2)); axes = axes.ravel()
for ax, mtr in zip(axes, box_metrics):
    lower = not (mtr in ('AUPRC', 'Coverage_90'))
    order = gm[mtr].mean().sort_values(ascending=lower).index.tolist()
    data = [raw.loc[raw.model == m, mtr].dropna().values for m in order]
    bp = ax.boxplot(data, patch_artist=True, widths=0.62, showmeans=True,
                    medianprops=dict(color='#333'), meanprops=dict(marker='D', markersize=4, markerfacecolor='#222'))
    for i, bx in enumerate(bp['boxes']): bx.set_facecolor(_bar_color(order[i])); bx.set_alpha(0.6)
    ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order, rotation=40, ha='right', fontsize=7)
    ax.set_title(mtr, fontsize=10)
for ax in axes[len(box_metrics):]: ax.set_visible(False)
fig.suptitle('Per-fold metric distributions (CV)', fontsize=12, y=1.01); fig.tight_layout()
fig.savefig(os.path.join(FIGS, '3_0_metric_boxes.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(FIGS, '3_0_metric_boxes.pdf'), bbox_inches='tight')
display(fig)

## 3. Significance: object-cluster bootstrap CI + Wilcoxon/Holm

CIs and paired tests use **pooled OOF** with bootstrap over site clusters (not naive √n_folds).

In [ ]:
from liquefaction_ai.evaluation import paired_significance, object_cluster_bootstrap, forest_plot
ocb = object_cluster_bootstrap(samples, ref=REF, nboot=2000)
ocb.to_csv(os.path.join(TABLES, 'significance_grouped_cluster_bootstrap.csv'), index=False)
pair = paired_significance(samples, ref=REF)
pair.to_csv(os.path.join(TABLES, 'significance_grouped_pairwise.csv'), index=False)
loo_ocb = None
if loo_samples is not None:
    loo_ocb = object_cluster_bootstrap(loo_samples, ref=REF, nboot=2000)
    loo_ocb.to_csv(os.path.join(TABLES, 'significance_loo_cluster_bootstrap.csv'), index=False)
print('=== object-cluster bootstrap (95% CI) ==='); display(ocb.round(4))
_pc = [c for c in ['metric','model','n','median_diff','median_diff_lo','median_diff_hi','p_holm','significant_0.05'] if c in pair.columns]
print(f'=== paired Wilcoxon+Holm vs {REF} ==='); display(pair[_pc].round(4))
try:
    display(forest_plot(ocb, metric='AUPRC', higher_better=True, ref=REF, out_dir=FIGS))
    if (ocb.get('metric') == 'traj_rmse_continuation').any() or 'traj_rmse_continuation' in ocb.columns:
        display(forest_plot(ocb, metric='traj_rmse_continuation', higher_better=False, ref=REF,
                            title='Post-prefix trajectory RMSE', out_dir=FIGS))
except Exception as e:
    print('forest_plot:', type(e).__name__, e)

## 4. Calibration & coverage (OOF)

Reliability diagram over pooled OOF + empirical site-held-out coverage@90 with CI (not a finite-sample guarantee).

In [ ]:
from liquefaction_ai.evaluation import reliability_diagram
fig, ece = reliability_diagram(samples, ref=REF, bins=10, out_dir=FIGS, suffix='cv'); print('ECE(OOF) =', round(ece, 4)); display(fig)
_cc = [c for c in ['model','Coverage_emp','Coverage_lo','Coverage_hi','mean_band_width','n_objects'] if c in objconf.columns]
print('=== empirical site-held-out coverage@90 ==='); display(objconf[_cc].round(3))

## 5. P³ / Pareto (physical admissibility)

Structured models are projected to monotone / ru≤1 (PVR=0) → admissible; flexible baselines may
violate physics. P³ ranks within the admissible set.

In [ ]:
from liquefaction_ai.evaluation import publication_ranking_table, pareto_plot, headline_table
from liquefaction_ai.evaluation.p3_ranking import P3_PROTOCOL_VERSION
permodel = primary_raw.groupby('model', as_index=False).mean(numeric_only=True)
p3 = publication_ranking_table(permodel, REF)
p3.to_csv(os.path.join(PUB, 'p3_core_ranking.csv'), index=False)
display(p3.round(3))
display(pareto_plot(summary, x='Traj_RMSE_continuation', y='AUPRC', ref=REF, out_dir=FIGS))
cluster = ocb if ocb is not None and len(ocb) else None
hl = headline_table(summary, cluster_df=cluster); hl.to_csv(os.path.join(PUB, 'publication_headline_grouped.csv'), index=False)
print('=== headline table (CI from cluster bootstrap) ==='); display(hl)
if loo_summary is not None:
    loo_permodel = loo_raw.groupby('model', as_index=False).mean(numeric_only=True)
    loo_p3 = publication_ranking_table(loo_permodel, REF)
    loo_p3.to_csv(os.path.join(TABLES, 'p3_core_ranking_loo.csv'), index=False)
    loo_hl = headline_table(loo_summary, cluster_df=loo_ocb)
    loo_hl.to_csv(os.path.join(PUB, 'publication_headline_loo.csv'), index=False)
    print('=== LOO headline ==='); display(loo_hl)

## 6. Breakdown by loading regime and soil type (OOF)

From per-sample OOF: how the error distributes over `regime` (liq/stable/unfinished) and soil type.

In [ ]:
smp = samples.copy()
assert 'soil_type' in smp.columns, 'sample-level soil_type was not propagated by compute_metrics'
def _by(group_col, metric='traj_rmse_continuation'):
    if group_col not in smp.columns: print('no column', group_col); return
    t = (smp.dropna(subset=[metric]).groupby(['model', group_col])[metric].mean().unstack().round(4))
    print(f'=== {metric} by {group_col} ==='); display(t)
_by('regime', 'traj_rmse_continuation')
_by('regime', 'coverage90')
if 'soil_type' in smp.columns: _by('soil_type', 'traj_rmse_continuation')

## 7. DPI-Flow ablations (component contribution, all folds)

Each component is switched off in turn; contribution = metric degradation. Computed across all folds.

In [ ]:
if RUN_ABLATIONS:
    from liquefaction_ai.evaluation.ablation_study import (run_ablation_fold, aggregate_ablations,
                                                              paired_ablation_summary)
    from liquefaction_ai.evaluation import ablation_bars
    feat_names = json.load(open('data/dataset/feature_names.json'))['static_feature_names']
    abl_folds = build_folds(meta, config, seed=42, loo=False, n_splits=N_SPLITS)  # 1 repeat for ablations
    def _dpi_flow_kwargs(fold_id):
        hit = primary_raw[(primary_raw['fold'] == fold_id) & (primary_raw['model'] == 'DPI-Flow')]
        if len(hit) != 1: raise RuntimeError(f'expected one DPI-Flow row for fold {fold_id}, got {len(hit)}')
        return json.loads(hit.iloc[0]['selected_model_kwargs'])
    rows = []
    for ab_seed in ABLATION_SEEDS:
        for k, ab_fold in enumerate(abl_folds):
            rows.append(run_ablation_fold(pop, config, ab_fold, k, feat_names, device, quick=QUICK,
                                            seed=ab_seed, base_kwargs=_dpi_flow_kwargs(k)))
    abl = pd.concat(rows, ignore_index=True); abl.to_csv(os.path.join(TABLES, 'ablations.csv'), index=False)
    abl_sum = aggregate_ablations(abl); abl_sum.to_csv(os.path.join(TABLES, 'ablations_summary.csv'), index=False)
    abl_pair = paired_ablation_summary(abl)
    abl_pair.to_csv(os.path.join(TABLES, 'ablations_paired_equivalence.csv'), index=False)
    display(abl_sum[[c for c in ['ablation','n_folds','n_seeds','n_runs','Traj_RMSE_worst_mean','AUPRC_mean','ECE_mean',
                                 'Physics_Violation_Rate_mean','N_liq_logMAE_mean'] if c in abl_sum.columns]].round(4))
    display(ablation_bars(abl_sum, metric='Traj_RMSE_worst', higher_better=False, out_dir=FIGS))
    display(ablation_bars(abl_sum, metric='AUPRC', higher_better=True, out_dir=FIGS))
    display(abl_pair.round(4))
else:
    print('RUN_ABLATIONS=False')

## 8. OOD stress: no-prefix robustness (fold 0)

The prefix-conditioned setting is stress-tested by zeroing the observed PPR prefix. A robust model
should degrade gracefully; a model that leans on the prefix collapses. Models are trained on the fold-0
train and evaluated with vs without the prefix (this needs per-fold models, so it is a dedicated pass).

In [ ]:
if RUN_OOD:
    from liquefaction_ai import prepare_benchmark_dataset
    from liquefaction_ai.evaluation.cross_validation import _train_one, publication_model_kwargs
    from liquefaction_ai.evaluation import collect_outputs, compute_metrics
    from liquefaction_ai.evaluation.metrics import stress_split
    from liquefaction_ai.training.loop import train_model
    from liquefaction_ai.training.persistence import load_model_metadata
    from liquefaction_ai.config import set_global_seed
    from liquefaction_ai import models as _M
    bench0 = prepare_benchmark_dataset(pop, config, device, precomputed_split=folds[0])
    ood_models = [('dpi_flow','DPI-Flow',True,'torch'), ('dpi_evt','DPI-EVT',True,'torch'),
                  ('evt_ssm','EVT-NeuralSSM',True,'torch'), ('transformer','Transformer',False,'torch')]
    def _ood_train(name, disp, isphys, trainer, train, val):
        # QUICK -> demo-эпохи (как evaluate_fold); иначе полноценное обучение через _train_one.
        if QUICK and trainer == 'torch':
            hp, _ = load_model_metadata('models', name); cls = getattr(_M, hp['model_type'])
            set_global_seed(config.seed); _kw = publication_model_kwargs(name, hp['model_kwargs'], config)
            m = cls(**_kw).to(device)
            ep = config.physics_epochs if isphys else config.baseline_epochs
            m, _ = train_model(m, train, val, epochs=ep, model_name=disp, config=config, device=device,
                               verbose=False, scheduler='cosine' if isphys else 'none')
            return m
        return _train_one(name, disp, isphys, trainer, train, val, config, device, 0, config.seed,
                          'models', nested=NESTED)
    rows = []
    for name, disp, isphys, trainer in ood_models:
        m = _ood_train(name, disp, isphys, trainer, bench0['train'], bench0['val'])
        for label, sp in [('with_prefix', bench0['test']),
                          ('no_prefix', stress_split(bench0['test'], no_prefix=True, drop_derived_aux=True))]:
            met, _ = compute_metrics(disp, collect_outputs(m, sp, config, device), sp, config)
            rows.append({'model': disp, 'stress': label, 'Traj_RMSE_continuation': met['Traj_RMSE_continuation'],
                         'AUPRC': met['AUPRC'], 'N_liq_logMAE': met['N_liq_logMAE']})
    ood = pd.DataFrame(rows); ood.to_csv(os.path.join(TABLES, 'ood_no_prefix.csv'), index=False)
    piv = ood.pivot(index='model', columns='stress', values='AUPRC')
    piv['AUPRC_drop'] = (piv['with_prefix'] - piv['no_prefix']).round(4)
    print('=== onset AUPRC: with vs no prefix (smaller drop = more robust) ==='); display(piv.round(4))
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4)); axes = axes.ravel()
    for ax, mtr, lower in zip(axes, ['Traj_RMSE_continuation', 'AUPRC', 'N_liq_logMAE'], [True, False, True]):
        p = ood.pivot(index='model', columns='stress', values=mtr).reindex(columns=['with_prefix', 'no_prefix'])
        order = p['with_prefix'].sort_values(ascending=lower).index.tolist(); xs = np.arange(len(order)); w = 0.38
        ax.bar(xs - w/2, p.loc[order, 'with_prefix'], w, label='with prefix', color='#2f7ab5')
        ax.bar(xs + w/2, p.loc[order, 'no_prefix'], w, label='no prefix (OOD)', color='#c98a3a')
        ax.set_xticks(xs); ax.set_xticklabels(order, rotation=30, ha='right', fontsize=8); ax.set_title(mtr, fontsize=10)
    axes[0].legend(fontsize=8, frameon=False)
    fig.suptitle('OOD no-prefix stress (fold 0): prefix-conditioned vs prefix-ablated', fontsize=12, y=1.03)
    fig.tight_layout(); fig.savefig(os.path.join(FIGS, '3_0_ood_no_prefix.png'), dpi=300, bbox_inches='tight')
    fig.savefig(os.path.join(FIGS, '3_0_ood_no_prefix.pdf'), bbox_inches='tight'); display(fig)
else:
    print('RUN_OOD=False')

## 9. A/B: conditional flow vs Gaussian posterior (pooled OOF)

Isolates the value of the conditional-flow posterior: identical backbone/schedule/seed per fold, only
the posterior family differs (flow vs diagonal Gaussian). Difference `gaussian − flow` with site-bootstrap CI.

In [ ]:
if RUN_AB:
    from liquefaction_ai import prepare_benchmark_dataset
    from liquefaction_ai.evaluation import ab_flow_vs_gaussian_pooled
    from liquefaction_ai.evaluation.ab_test import train_ab_pair
    ab_folds = build_folds(meta, config, seed=42, loo=False, n_splits=N_SPLITS, n_repeats=1)
    pairs = []
    for k, fold in enumerate(ab_folds):
        bench_ab = prepare_benchmark_dataset(pop, config, device, precomputed_split=fold)
        mf, mg = train_ab_pair(bench_ab, config, device, _dpi_flow_kwargs(k), seed=config.seed + k,
                               mc_train_samples=4, mc_crps_weight=0.3)
        pairs.append((mf, mg, bench_ab['test'])); print('A/B fold', k, 'done')
    ab = ab_flow_vs_gaussian_pooled(pairs, config, device, n_samples=128, nboot=5000, seed=42)
    ab.to_csv(os.path.join(TABLES, 'ab_flow_vs_gaussian_pooled.csv'), index=False)
    d = ab.dropna(subset=['diff_gauss_minus_flow']).reset_index(drop=True)
    figw, fig = new_figure((7.2, 2.7)); ax = fig.add_subplot(111); yy = list(range(len(d)))
    ax.errorbar(d['diff_gauss_minus_flow'], yy,
                xerr=[d['diff_gauss_minus_flow'] - d['ci95_low'], d['ci95_high'] - d['diff_gauss_minus_flow']],
                fmt='o', color='#2f7ab5', capsize=4)
    ax.axvline(0.0, ls='--', color='#c46b6b', lw=1.2); ax.set_yticks(yy); ax.set_yticklabels(d['metric'])
    ax.set_xlabel('gaussian - flow  (>0 -> flow better; lower NLL/CRPS = better)')
    ax.set_title('A/B: conditional flow vs Gaussian posterior (pooled OOF, site-bootstrap 95% CI)')
    fig.savefig(os.path.join(FIGS, '3_0_ab_flow_vs_gaussian.png'), dpi=300, bbox_inches='tight')
    fig.savefig(os.path.join(FIGS, '3_0_ab_flow_vs_gaussian.pdf'), bbox_inches='tight')
    display(ab.round(4))
else:
    print('RUN_AB=False (heavy; enable to quantify the flow-posterior contribution)')

## 10. Consistency, P³ sensitivity & reproducibility manifest

In [ ]:
from liquefaction_ai.data.consistency import check_artifact_consistency
artifact_ok, artifact_report = check_artifact_consistency('data/dataset')
if not artifact_ok: raise RuntimeError('artifact consistency failed:\n' + '\n'.join(artifact_report))
print('artifact consistency: OK\n' + '\n'.join(artifact_report))

# P3 robustness to the choice of reference model: physics models must stay admissible under any reference.
for ref_alt in ['DPI-Flow', 'DPI-EVT', 'PINN']:
    if ref_alt in permodel.model.values:
        t = publication_ranking_table(permodel, ref_alt)
        if 'excluded_from_admissible_ranking' in t.columns:
            adm = t.loc[~t['excluded_from_admissible_ranking'].astype(bool), 'model'].tolist()
        else:
            adm = t['model'].tolist()
        print(f'ref={ref_alt}: admissible =', adm)

# Manifest records the ACTUAL fold-specific nested choices, not only global training artifacts.
from liquefaction_ai.evaluation import write_run_manifest, validate_run_manifest
_hp_cols = ['repeat', 'fold', 'test_object', 'model', 'P3_reference',
            'nested_selection_applied', 'selected_model_kwargs']
_hp_frames = [raw.assign(protocol='grouped')]
if loo_raw is not None: _hp_frames.append(loo_raw.assign(protocol='loo'))
fold_hyperparameters = pd.concat(_hp_frames, ignore_index=True)[['protocol'] + _hp_cols]
fold_hyperparameters = json.loads(fold_hyperparameters.to_json(orient='records'))
run_metadata = dict(publication_run=PUBLICATION_RUN, quick=QUICK, nested=NESTED,
                    run_loo=RUN_LOO, run_ablations=RUN_ABLATIONS, run_ood=RUN_OOD, run_ab=RUN_AB,
                    n_splits=N_SPLITS, n_repeats=N_REPEATS, ablation_seeds=list(ABLATION_SEEDS),
                    p3_reference=REF, p3_protocol_version=P3_PROTOCOL_VERSION, device=str(device),
                    output_root=os.path.relpath(RUN_ROOT, REPO),
                    source_git_clean=SOURCE_GIT_CLEAN)
p = write_run_manifest(pop, config, REPO, models_dir='models',
                       out_path=os.path.join(RUN_ROOT, 'run_manifest.json'),
                       require_clean=False,  # cleanliness was checked before training
                       run_metadata=run_metadata, fold_hyperparameters=fold_hyperparameters)
if PUBLICATION_RUN:
    manifest = json.load(open(p))
    validate_run_manifest(manifest,
        required_models=('dpi_flow', 'dpi_evt', 'evt_ssm'),
        required_results=('cv_grouped_raw.csv', 'cv_loo_raw.csv', 'publication_headline_grouped.csv',
                          'publication_headline_loo.csv', 'p3_core_ranking.csv', 'nliq_tail_cases.csv',
                          'ablations.csv', 'ablations_paired_equivalence.csv',
                          'ood_no_prefix.csv', 'ab_flow_vs_gaussian_pooled.csv'),
        require_fold_hyperparameters=True)
print('run manifest ->', p)

## Summary

- **Headline** = §1 table (fold mean ± SD) + §2 figures + §3 cluster-bootstrap CI + §5 P³/Pareto.
- PVR audits strict monotonicity and `0≤ru≤1` over the full forecast grid; structural zero is a guarantee, not evidence.
- The fold spread (§2) is the real uncertainty on a small cohort — **not** a single split.
- §8 OOD stress and §9 flow-vs-Gaussian A/B isolate robustness and the flow-posterior contribution.
- Publication settings are validated before training; smoke artifacts cannot overwrite headline tables.
